<div text-align="center">
  <img src="https://raw.githubusercontent.com/FarnoushRJ/MambaLRP/main/assets/MambaLRP_logo.jpeg" width="1000"/>
</div>


<div text-align="center"><h1>🐍 MambaLRP is here! 🎉</h1>

Clone the repository and install MambaLRP.

In [1]:
# Cell 1: Clone
!git clone https://github.com/AdamBosch/MambaLRP.git || echo "Already cloned"

fatal: destination path 'MambaLRP' already exists and is not an empty directory.
Already cloned


In [1]:
# Cell 2: Pin torch first
!pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 torchaudio==2.1.0+cu118 \
    --index-url https://download.pytorch.org/whl/cu118 -q


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
import subprocess
result = subprocess.run(['find', '/home/jovyan', '-name', 'torch', '-type', 'd'], 
                       capture_output=True, text=True)
print(result.stdout)

/home/jovyan/.local/lib/python3.10/site-packages/torch
/home/jovyan/.local/lib/python3.10/site-packages/torch/include/torch
/home/jovyan/.local/lib/python3.10/site-packages/torch/include/torch/csrc/api/include/torch



In [3]:
# Alternative Cell 3 - use pip's --find-links and pre-built wheels
!PYTHONPATH=/home/jovyan/.local/lib/python3.10/site-packages \
    pip install causal-conv1d==1.2.0.post2 --no-build-isolation --no-cache-dir -q
!PYTHONPATH=/home/jovyan/.local/lib/python3.10/site-packages \
    pip install mamba-ssm==1.2.0.post1 --no-build-isolation --no-cache-dir -q


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [4]:
# Cell 4: Write constraints file and install the rest
!echo "torch==2.1.0+cu118" > /tmp/constraints.txt
!pip install "numpy<2.0.0" "pyarrow<15.0.0" datasets==2.14.5 \
    transformers==4.40.1 captum==0.7.0 einops \
    -c /tmp/constraints.txt -q


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip


In [5]:
# Cell 5: Install MambaLRP
!pip install ./MambaLRP --no-deps -q


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip


In [6]:
import torch
print(torch.__version__)       # should be 2.1.0+cu118
import causal_conv1d
import mamba_ssm
print("All imports OK")

2.1.0+cu118
All imports OK


In [7]:
from transformers import MambaConfig, MambaForCausalLM, AutoTokenizer
import sys

from mamba_lrp.model.mamba_huggingface import ModifiedMambaForCausalLM
from mamba_lrp.model.utils import *
from mamba_lrp.lrp.utils import relevance_propagation
from mamba_lrp.dataset.general_dataset import get_sst_dataset, get_medbios_dataset
import torch
import numpy as np

## Load model

Load model and tokenizer.

In [8]:
import torch
import gc

# 1. Delete the variables holding the references
# Note: Ensure you are actually deleting the correct names
if 'R' in locals(): del R
if 'attr' in locals(): del attr
if 'embeddings' in locals(): del embeddings
if 'morf_probs' in locals(): del morf_probs

# 2. Force Python to clear unused references from RAM
gc.collect()

# 3. Force PyTorch to release its internal cache back to the OS
torch.cuda.empty_cache()

# 4. Optional: Check how much is actually used vs reserved
print(f"Actually Used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved (Cached): {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Actually Used: 0.00 GB
Reserved (Cached): 0.00 GB


In [10]:
!pip install gdown

# Import gdown
import gdown

# Define the file ID and the destination file name
file_id = '12ehupwHXy80_ymyKVwPMTSpyXNTN6cmA'  # MedBios
destination = 'mamba_medbios_weights.pt'
# Construct the URL
url = f'https://drive.google.com/uc?id={file_id}'

# Download the file
gdown.download(url, destination, quiet=False)

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip


Downloading...
From (original): https://drive.google.com/uc?id=12ehupwHXy80_ymyKVwPMTSpyXNTN6cmA
From (redirected): https://drive.google.com/uc?id=12ehupwHXy80_ymyKVwPMTSpyXNTN6cmA&confirm=t&uuid=36b1207b-04ac-42d3-b815-0219b18b9efa
To: /home/jovyan/mamba_medbios_weights.pt
100%|██████████| 517M/517M [00:03<00:00, 169MB/s]  


'mamba_medbios_weights.pt'

In [9]:
# Initialize tokenizer and base model
tokenizer = AutoTokenizer.from_pretrained("state-spaces/mamba-130m-hf")
tokenizer.eos_token = "<|endoftext|>"
tokenizer.bos_token = "<|startoftext|>"
tokenizer.pad_token = "<|pad|>"
tokenizer.unk_token = "<|unkown|>"
tokenizer.add_tokens(['<|unkown|>', '<|pad|>', "<|startoftext|>"], special_tokens=True)

model = MambaForCausalLM.from_pretrained("state-spaces/mamba-130m-hf", use_cache=True)
resize_token_embeddings(model, len(tokenizer))
model.lm_head = torch.nn.Linear(768, 28, bias=True)

/home/jovyan/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [11]:
import torch
from torch.utils.data import DataLoader
from transformers import AdamW, AutoTokenizer
from datasets import load_dataset
from torch.optim.lr_scheduler import LinearLR
from tqdm import tqdm

def train_snli_mamba(model, tokenizer, device):
    # 1. Dataset Setup (SNLI from Hugging Face)
    dataset = load_dataset("snli")
    
    def tokenize_function(examples):
        # Concatenate Premise and Hypothesis as per standard NLI practice
        return tokenizer(
            examples['premise'], 
            examples['hypothesis'], 
            truncation=True, 
            padding='max_length', 
            max_length=128
        )

    # Filter out samples with label -1 (unlabeled/noisy in SNLI)
    dataset = dataset.filter(lambda x: x['label'] != -1)
    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    tokenized_dataset.set_format(type='torch', columns=['input_ids', 'label'])
    
    train_loader = DataLoader(tokenized_dataset['train'], batch_size=64, shuffle=True)

    # 2. FREEZE BACKBONE 
    # Keeping pretrained parameters fixed as per MambaLRP methodology
    for param in model.parameters():
        param.requires_grad = False
    
    # 3. SETUP CLASSIFICATION HEAD
    # Mapping the hidden state (768) to 3 classes (Entailment, Neutral, Contradiction)
    model.lm_head = torch.nn.Linear(768, 3, bias=True).to(device)
    
    optimizer = AdamW(model.lm_head.parameters(), lr=7e-5)
    criterion = torch.nn.CrossEntropyLoss()
    scheduler = LinearLR(optimizer, start_factor=0.5, total_iters=5)

    print(f"Starting SNLI training: 5 epochs, {len(train_loader)} batches per epoch.")
    
    for epoch in range(5):
        model.train()
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        
        for batch in pbar:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(input_ids)
            # Use the logit from the last token in the sequence
            logits = outputs.logits[:, -1, :] 
            
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        print(f"Epoch {epoch+1} Complete. Avg Loss: {total_loss / len(train_loader):.4f}")
        scheduler.step()

    torch.save(model.state_dict(), 'mamba_snli_weights.pt')
    print("Fine-tuned SNLI weights saved.")
    
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device) # Move entire model to GPU first
train_snli_mamba(model, tokenizer, device)

Extracting data files:   0%|          | 0/3 [00:00<?, ?it/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/550152 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/549367 [00:00<?, ? examples/s]

/home/jovyan/.local/lib/python3.10/site-packages/transformers/optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Starting SNLI training: 5 epochs, 17168 batches per epoch.


Epoch 1:   1%|          | 143/17168 [00:21<43:23,  6.54it/s, loss=1.1090]


KeyboardInterrupt: 

In [12]:
model.load_state_dict(
    torch.load('mamba_medbios_weights.pt', map_location=torch.device('cpu')),
    strict=True
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

# Make model explainable.
modified_model = ModifiedMambaForCausalLM(model, is_fast_forward_available=False)
modified_model.eval()
model.backbone.embeddings.requires_grad = False
pretrained_embeddings = model.backbone.embeddings

## Load dataset

Load SST-2 dataset.

In [13]:
validation_dataset = get_medbios_dataset(
    tokenizer=tokenizer,
    truncation=False,
    max_length=None
    )

## Generate explanation

Generate explanation for one sample.

In [14]:
i = 413
input_ids = validation_dataset.__getitem__(i)['input_ids'].unsqueeze(0).to(device)
label = torch.tensor(validation_dataset.__getitem__(i)['label']).long().to(device)
idx = torch.where(input_ids == 0)[1] + 1
input_ids = input_ids[:, :idx]
embeddings = pretrained_embeddings(input_ids)

R, prediction = relevance_propagation(
    model=modified_model,
    embeddings=embeddings,
    targets=label,
    n_classes=28
    )

## Visualization

For simplicity, we use the visualization utilities in Captum to display the results.

In [15]:
from captum.attr import visualization as viz

In [16]:
tokens = []
for id in input_ids[0][1: -2]:
    tokens.append(tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens([id.item()])))
attributions = R[0][1: -2]
attributions = attributions / attributions.max()

In [17]:
# Visualize the attributions
viz.visualize_text([viz.VisualizationDataRecord(
    attributions,
    torch.max(model(input_ids).logits[:, -1, :], dim=1).values.item(),
    torch.argmax(model(input_ids).logits[:, -1, :], dim=1).item(),
    true_class=label.item(),
    attr_class=label.item(),
    attr_score=attributions.sum(),
    raw_input_ids=tokens,
    convergence_score=None
)])

In [18]:
if isinstance(attributions, np.ndarray):
    attributions_tensor = torch.from_numpy(attributions)
else:
    attributions_tensor = attributions
    

sorted_indices = torch.argsort(attributions_tensor, descending=True)

In [19]:
def get_morf_curve_logits(model, input_ids, sorted_indices, target_label):
    scores = []
    temp_input_ids = input_ids.clone()
    
    with torch.no_grad():
        logits = model(temp_input_ids) 
        initial_logit = logits[0, -1, target_label].item()
        scores.append(initial_logit)

    for idx in sorted_indices:
        temp_input_ids[0, idx + 1] = tokenizer.pad_token_id 
        
        with torch.no_grad():
            logits = model(temp_input_ids)
            logit = logits[0, -1, target_label].item()
            scores.append(logit)
            
    return scores
#morf_values = get_morf_curve_logits(modified_model, input_ids, sorted_indices, label.item())

In [20]:
def get_morf_curve_batched(model, input_ids, embeddings, sorted_indices, target_label, chunk_size=8):
    seq_len = len(sorted_indices)
    
    # Work in embedding space, not token ID space
    base_embeddings = embeddings.clone().repeat(seq_len + 1, 1, 1)  # [N+1, seq_len, hidden]
    
    for i, idx in enumerate(sorted_indices):
        # Zero out the embedding at position idx+1 for all subsequent steps
        base_embeddings[i+1:, idx + 1, :] = 0.0
    
    all_logits = []
    with torch.no_grad():
        for i in range(0, base_embeddings.size(0), chunk_size):
            chunk = base_embeddings[i:i+chunk_size].to(device)
            logits = model(inputs_embeds=chunk)  # pass embeddings directly
            all_logits.extend(logits[:, -1, target_label].cpu().tolist())
            del chunk, logits
    return all_logits

In [21]:
import numpy as np
from tqdm import tqdm

num_samples = 50
all_delta_afs = []

print(f"Evaluating {num_samples} samples from Medical-Bios...")

for i in tqdm(range(num_samples)):
    sample = validation_dataset.__getitem__(i)
    input_ids = sample['input_ids'].unsqueeze(0).to(device)
    label = torch.tensor(sample['label']).long().to(device)
    
    idx = torch.where(input_ids == 0)[1] + 1
    input_ids = input_ids[:, :idx]
    
    # Get pred_label PER SAMPLE
    with torch.no_grad():
        outputs = model(input_ids)
        pred_label = torch.argmax(outputs.logits[0, -1, :]).item()
    
    embeddings = pretrained_embeddings(input_ids)
    R, _ = relevance_propagation(
        model=modified_model, 
        embeddings=embeddings,
        targets=torch.tensor(pred_label).long().to(device), 
        n_classes=28
    )
    
    attr = R[0][1:-1]
    if isinstance(attr, np.ndarray):
        attr = torch.from_numpy(attr)
    
    idx_morf = torch.argsort(attr.abs(), descending=True)[:5]
    morf_curve = get_morf_curve_batched(modified_model, input_ids, embeddings, idx_morf, pred_label)
    af_morf = np.trapz(morf_curve) / (len(morf_curve) - 1)
    
    idx_lerf = torch.argsort(attr.abs(), descending=False)[:5]
    lerf_curve = get_morf_curve_batched(modified_model, input_ids, embeddings, idx_lerf, pred_label)
    af_lerf = np.trapz(lerf_curve) / (len(lerf_curve) - 1)
    
    # delta_af = af_morf - af_lerf
    delta_af = af_lerf - af_morf
    all_delta_afs.append(delta_af)
    # del R, embeddings, input_ids, attr, morf_curve, lerf_curve
    # gc.collect()
    # if i % 10 == 0:
    #     torch.cuda.empty_cache()
    

# Final Result
mean_delta_af = np.mean(all_delta_afs)
print(f"\nMean Delta AF over {num_samples} samples: {mean_delta_af:.4f}")
print(f"Paper Reference (Mamba-130M Medical-Bios): 3.906")

Evaluating 50 samples from Medical-Bios...


100%|██████████| 50/50 [00:51<00:00,  1.03s/it]


Mean Delta AF over 50 samples: 0.0145
Paper Reference (Mamba-130M Medical-Bios): 3.906
